# CreatorOps Video Agent

This notebook runs the same video workflow as the Streamlit frontend: download or load a source, trim it, add an optional overlay and music, then export an MP4.

Use media you own or have permission to edit.

In [ ]:
from pathlib import Path
import sys
import tempfile

PROJECT_ROOT = Path.cwd().resolve().parent
if PROJECT_ROOT.name != 'CreatorAgent':
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Backend.video_processor import build_video, download_youtube, save_upload

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Choose one source. Leave YOUTUBE_URL empty to use LOCAL_SOURCE.
YOUTUBE_URL = ''
LOCAL_SOURCE = PROJECT_ROOT / 'media' / 'source.mp4'

OVERLAY_VIDEO = None  # Example: PROJECT_ROOT / 'media' / 'overlay.mp4'
BACKGROUND_MUSIC = None  # Example: PROJECT_ROOT / 'media' / 'music.mp3'

START_SECONDS = 0.0
END_SECONDS = None  # Set a number such as 30.0, or leave None for the full source.
RATIO = '9:16'  # '9:16', '16:9', or '1:1'
OVERLAY_POSITION = 'Top right'  # 'Top right', 'Bottom right', or 'Bottom left'
MUSIC_VOLUME = 0.18

In [ ]:
# Prepare the source and optional layers. This cell downloads only when YOUTUBE_URL is set.
with tempfile.TemporaryDirectory(prefix='creatorops-notebook-') as temporary_dir:
    work_dir = Path(temporary_dir)
    if YOUTUBE_URL.strip():
        source = download_youtube(YOUTUBE_URL.strip(), work_dir)
    else:
        source = Path(LOCAL_SOURCE)
        if not source.exists():
            raise FileNotFoundError(f'Source video not found: {source}')
    overlay = Path(OVERLAY_VIDEO) if OVERLAY_VIDEO else None
    music = Path(BACKGROUND_MUSIC) if BACKGROUND_MUSIC else None
    output = PROJECT_ROOT / 'media' / 'creatorops-notebook-final.mp4'
    output.parent.mkdir(parents=True, exist_ok=True)
    build_video(source, overlay, music, output, START_SECONDS, END_SECONDS, RATIO, OVERLAY_POSITION, MUSIC_VOLUME)

print(f'Finished: {output}')

In [ ]:
from IPython.display import Video, display
display(Video(str(output), embed=True))